> ## SOLUTION / ANSWER KEY &mdash; Lab 4.5
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-05-dataset-prep.ipynb`](../lab-05-dataset-prep.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 4.5 &mdash; Preparing a Dataset for Fine-tuning (tokenized & ready)

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Separate raw examples into texts and integer labels
- Tokenize the whole set into a padded tensor batch
- Produce the (encodings, labels) a training loop needs

> **How this lab works (near-real):** these labs use **real Hugging Face Transformers** locally on CPU &mdash; a real pretrained sentiment model, a real tokenizer, and (the headline) a **real fine-tune** you run yourself. Read the **Concept**, fill the real `___` blanks in **Build it** (real model / tokenizer / training-loop calls), **Run it for real** to see the **actual model output** (including the real **before &rarr; after** fine-tune numbers), note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real results you can read. The genuine maths (softmax, precision/recall) you still compute **by hand** &mdash; real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased-finetuned-sst-2-english` (out-of-the-box sentiment + logits), `distilbert-base-uncased` (tokenizer), `prajjwal1/bert-tiny` (frozen features **and** the model you fine-tune). First use downloads the weights (needs network), then they are cached. An optional hosted comparison uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 4 slides &mdash; Dataset prep for fine-tuning](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (optional hosted compare)

WORK = "/tmp/biaa-lab-04-05"
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
Before fine-tuning you need clean **texts**, integer **labels**, and those texts **tokenized** into
tensors. Real datasets have string labels ("pos"/"neg") that you **encode** to integers, then you run
the real tokenizer over everything to get the `input_ids`/`attention_mask` batch plus a `labels`
tensor &mdash; the precise inputs the fine-tune loop in Lab 4.10 feeds to the model.

## Build it
Encode labels to integers, then tokenize the whole set into tensors.

In [ ]:
RAW = [("Loved it!", "pos"), ("So boring.", "neg"), ("Brilliant film", "pos"),
       ("A dull mess", "neg"), ("Wonderful!", "pos"), ("Terrible.", "neg"),
       ("Great stuff", "pos"), ("I hated it", "neg")]
label2id = {"neg": 0, "pos": 1}

import torch
from transformers import AutoTokenizer
def prepare(tok):
    texts = [t.lower().strip() for t, lbl in RAW]
    y = [label2id[lbl] for t, lbl in RAW]
    enc = tok(texts, padding=True, truncation=True, return_tensors="pt")
    return enc, torch.tensor(y)

## Run it for real
Run the real tokenizer and inspect the training-ready batch.

In [ ]:
try:
    from collections import Counter
    tok = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")
    enc, y = prepare(tok)
    print("input_ids shape:", tuple(enc["input_ids"].shape), "(examples, padded_length)")
    print("labels         :", y.tolist())
    print("class balance  :", dict(Counter(y.tolist())), " (0=neg, 1=pos)")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- `enc["input_ids"]` is a real `(8, L)` tensor &mdash; 8 examples padded to a common length `L`, ready for the model.
- The `labels` tensor is what the model compares its predictions against to compute the training **loss**.
- The **class balance** check matters: a lopsided set lets a model "win" by always predicting the majority. Prep is 80% of any fine-tune.

## Your turn (open task &mdash; no grader)
Add a few of your own rows (keep the balance even). Add a third class (e.g. a neutral label)
and extend `label2id` &mdash; does everything downstream still line up? A "good" answer: you can point
to the `(examples, length)` shape and say why every row must share the same length.

---
### Key takeaway
Clean text + integer labels + a tokenized tensor batch is exactly what a fine-tune loop eats. Garbage in, garbage out &mdash; prep earns the accuracy.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>